# Query L2 VM2 Uniform Data Using Redivis Python API

For more details on the L2 VM2 Uniform dataset, please see [this documentation](https://rs-kellogg.github.io/krs-public-docs-prototype/services/datasets/datasets/l2-vm2-uniform.html).

## Import Python Packages

In [ ]:
import redivis

## Connect to Redivis and to the L2 Dataset

In [ ]:
# If you do not have a Redivis API Token (https://docs.redivis.com/api/rest-api/authorization#api-token)
# then when you run this line, you will be asked to authenticate to allow the Python API to work
# the authentication will last for a few hours before it will ask you to authenticate again.
organization = redivis.organization("Northwestern")

# Specify which dataset we are using
dataset = organization.dataset("l2_voter_data:azqd")

## Perform a query

On Redivis, the L2 VM2 Uniform data is organized as one table per state, named using the two-letter state abbreviation followed by _Uniform — for example `AK_Uniform`, `AL_Uniform`, `AR_Uniform`, … `WY_Uniform`. Each table has the same ~800-column schema. To work with multiple states, use UNION ALL across the relevant state tables (see the multi-state query example below).

### Count registered voters by party in a single state

In [ ]:
# Execute your Redivis query
query = dataset.query("""
SELECT
    Parties_Description  AS party,
    COUNT(*)             AS voter_count
FROM IL_Uniform  -- replace with your target state table
WHERE Voters_Active = 'A'
GROUP BY 1
ORDER BY 2 DESC;
""")

In [ ]:
# Convert to pandas DataFrame
df = query.to_pandas_dataframe()

print(df)

### Identify consistent general-election voters (voted in all elections 2016–2024)

In [ ]:
# Execute your Redivis query
query = dataset.query("""
SELECT
    LALVOTERID,
    Voters_FirstName,
    Voters_LastName,
    Residence_Addresses_State,
    Parties_Description
FROM AK_Uniform
WHERE
    Voters_Active  = 'A'
    AND General_2024 = 'Y'
    AND General_2022 = 'Y'
    AND General_2020 = 'Y'
    AND General_2018 = 'Y'
    AND General_2016 = 'Y'
LIMIT 100;
""")

In [ ]:
# Convert to pandas DataFrame
df = query.to_pandas_dataframe()

print(df)

## Combine multiple states with UNION ALL

In [ ]:
# Execute your Redivis query
query = dataset.query("""
SELECT
    'IL' AS state, Parties_Description AS party, COUNT(*) AS voter_count
FROM IL_Uniform
WHERE Voters_Active = 'A'
GROUP BY 1, 2

UNION ALL

SELECT
    'IN' AS state, Parties_Description AS party, COUNT(*) AS voter_count
FROM IN_Uniform
WHERE Voters_Active = 'A'
GROUP BY 1, 2

ORDER BY 1, 3 DESC;
""")

In [ ]:
# Convert to pandas DataFrame
df = query.to_pandas_dataframe()

print(df)

### Age and income distribution of Democratic primary voters in Illinois

In [ ]:
# Execute your Redivis query
query = dataset.query("""
SELECT
    CASE
        WHEN CAST(Voters_Age AS INT64) BETWEEN 18 AND 29 THEN '18-29'
        WHEN CAST(Voters_Age AS INT64) BETWEEN 30 AND 44 THEN '30-44'
        WHEN CAST(Voters_Age AS INT64) BETWEEN 45 AND 64 THEN '45-64'
        WHEN CAST(Voters_Age AS INT64) >= 65            THEN '65+'
        ELSE 'Unknown'
    END                                             AS age_group,
    ConsumerData_Estimated_Income_Amount            AS income_range,
    COUNT(*)                                        AS voter_count
FROM IL_Uniform
WHERE
    Parties_Description = 'Democratic'
    AND Primary_2024    = 'Y'
GROUP BY 1, 2
ORDER BY 1, 3 DESC;
""")

In [ ]:
# Convert to pandas DataFrame
df = query.to_pandas_dataframe()

print(df)